# Buổi 28 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `phan_cap.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Cây phân cấp và ma trận S (mục 4.1)

In [ ]:
%matplotlib inline
import warnings

import numpy as np
import phan_cap as pc

warnings.simplefilter("ignore")
Y, S_df, tags = pc.tong_hop(pc.doc_du_lich())
ten, S = pc.ma_tran_S(S_df)
print("S:", S.shape, "| số chuỗi mỗi cấp:", {pc.TEN_CAP[k]: len(v) for k, v in tags.items()}, "| số quý:", Y["ds"].nunique())

## Bước 2 — Dự báo base và độ lệch cộng (mục 4.1–4.3)

In [ ]:
ra = pc.du_bao_mot_moc(Y, S_df, tags, pc.MOC_CAT[-1])
for c in ["AutoETS", "AutoETS/BottomUp", "AutoETS/MinTrace_method-ols", "AutoETS/MinTrace_method-mint_shrink"]:
    print(f"{c:40s} lệch cộng lớn nhất: {pc.do_lech_cong(ra, S_df, c):,.4g}")
du = pc.du_bao_khop(ra)
print("dự báo đem dùng (", pc.PHUONG_PHAP, ") lệch cộng:", round(pc.do_lech_cong(du.rename(columns={'du_bao': 'x'}), S_df, "x"), 4))

## Bước 3 — OLS tự viết so với thư viện (mục 4.3)

In [ ]:
base = ra.pivot(index="unique_id", columns="ds", values="AutoETS").loc[ten].to_numpy()
lib = ra.pivot(index="unique_id", columns="ds", values="AutoETS/MinTrace_method-ols").loc[ten].to_numpy()
print("lệch lớn nhất OLS tự viết − thư viện:", float(np.abs(pc.hoa_giai_ols(base, S) - lib).max()))

## Bước 4 — Backtest theo từng cấp (mục 4.2–4.3)

In [ ]:
R = pc.backtest(Y, S_df, tags)
cot = ["AutoETS", "SeasonalNaive", "AutoETS/BottomUp", "AutoETS/TopDown_method-forecast_proportions",
       "AutoETS/MinTrace_method-ols", "AutoETS/MinTrace_method-mint_shrink"]
B = pc.bang_sai_so(R, cot)
print(B.pivot(index="cap", columns="cach", values="rmse")[cot].round(1).to_string())
print(B.pivot(index="cap", columns="cach", values="sai_so_co_dau")[cot].round(1).to_string())

## Bước 5 — Khoảng khớp và calibration từng cấp (mục 4.4)

Khoảng 1–2 phút.

In [ ]:
for c in ["AutoETS", "AutoETS/MinTrace_method-mint_shrink"]:
    print("chuẩn, trong mẫu", c, pc.coverage(R, c).round(3).to_dict())
F, A = pc.sai_so_ngoai_mau(Y, S_df)
K = pc.khoang_khop(F, A, S)
cap = {i: pc.TEN_CAP[k] for k, v in tags.items() for i in v}
K["cap"] = K["unique_id"].map(cap)
K["trong"] = (K["y"] >= K["lo"]) & (K["y"] <= K["hi"])
print(K.groupby(["cap", "cach"]).trong.mean().unstack().round(3).to_string())

## Bước 6 — Phân cấp theo thời gian (mục 4.4, hộp Nâng cao)

In [ ]:
print(pc.phan_cap_thoi_gian(Y).round(1).to_string(index=False))

## Bước 7 — Kiểm tra

Trong terminal, thư mục `lab/`: `python lab.py check` — xanh 6/6 là xong.